<a href="https://colab.research.google.com/github/hugolo000/DigitalImage_PROJECT/blob/main/notebooks/03_baseline_model_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q -U segmentation-models-pytorch albumentations opencv-python-headless

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

# Define hyperparameters and model specs
ENCODER = 'resnet34'       # Pretrained backbone
ENCODER_WEIGHTS = 'imagenet'
NUM_CLASSES = 12           # CamVid (SegNet version): classes 0-10 + class 11 = unlabelled/void
VOID_CLASS = 11            # Ignored in the loss and in the metrics
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Instantiate the U-Net architecture with the selected backbone
model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=NUM_CLASSES
)

model = model.to(DEVICE)
print(f"Model successfully loaded on device: {DEVICE}")

Model successfully loaded on device: cpu


Data loading (CamVid) and DataLoaders

In [7]:
import zipfile
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

# 1. Download and extract CamVid (each Colab runtime starts empty)
DATASET_URL = "https://github.com/alexgkendall/SegNet-Tutorial/archive/refs/heads/master.zip"
DATA_DIR = Path("./data")
RAW_ZIP_PATH = DATA_DIR / "camvid_raw.zip"
EXTRACTED_DIR = DATA_DIR / "extracted"

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not RAW_ZIP_PATH.exists():
    urllib.request.urlretrieve(DATASET_URL, RAW_ZIP_PATH)
if not EXTRACTED_DIR.exists():
    with zipfile.ZipFile(RAW_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACTED_DIR)

BASE_DIR = (EXTRACTED_DIR / "SegNet-Tutorial-master").resolve()

def load_camvid_split(txt_filename: str, base_dir: Path) -> pd.DataFrame:
    """Parses a CamVid split file keeping only the parent folder and file name."""
    image_paths, mask_paths = [], []
    with open(base_dir / "CamVid" / txt_filename, "r") as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) == 2:
                img_p, mask_p = Path(parts[0]), Path(parts[1])
                img_abs = (base_dir / "CamVid" / img_p.parts[-2] / img_p.parts[-1]).resolve()
                mask_abs = (base_dir / "CamVid" / mask_p.parts[-2] / mask_p.parts[-1]).resolve()
                if img_abs.exists() and mask_abs.exists():
                    image_paths.append(str(img_abs))
                    mask_paths.append(str(mask_abs))
    return pd.DataFrame({"image_path": image_paths, "mask_path": mask_paths})

df_train = load_camvid_split("train.txt", BASE_DIR)
df_val = load_camvid_split("val.txt", BASE_DIR)
df_test = load_camvid_split("test.txt", BASE_DIR)
assert len(df_train) > 0, "df_train is empty: check the extracted folder structure"
print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

# 2. Preprocessing pipelines (same as notebook 02)
IMAGE_HEIGHT, IMAGE_WIDTH = 256, 256
MEAN = np.array([0.485, 0.456, 0.406])
STD = np.array([0.229, 0.224, 0.225])

train_transforms = A.Compose([
    A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(mean=tuple(MEAN), std=tuple(STD)),
    ToTensorV2()
], is_check_shapes=False)

valid_transforms = A.Compose([
    A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),
    A.Normalize(mean=tuple(MEAN), std=tuple(STD)),
    ToTensorV2()
], is_check_shapes=False)

# 3. Dataset: returns a normalized image tensor and a LONG mask tensor
class CamVidDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = cv2.cvtColor(cv2.imread(self.df.loc[idx, "image_path"]), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.df.loc[idx, "mask_path"], cv2.IMREAD_GRAYSCALE)
        out = self.transform(image=image, mask=mask)
        # CrossEntropy / Dice losses require class indices as int64
        return out["image"], out["mask"].long()

# 4. DataLoaders
BATCH_SIZE = 8
train_loader = DataLoader(CamVidDataset(df_train, train_transforms), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(CamVidDataset(df_val, valid_transforms), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2)
test_loader = DataLoader(CamVidDataset(df_test, valid_transforms), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)

# 5. Sanity check: shapes, dtypes and class index range
images, masks = next(iter(train_loader))
print(f"Images: {tuple(images.shape)} {images.dtype} | Masks: {tuple(masks.shape)} {masks.dtype}")
print(f"Unique mask values in batch: {masks.unique().tolist()}")
assert masks.max() < NUM_CLASSES, "Mask contains class indices >= NUM_CLASSES"

Train: 367 | Val: 101 | Test: 233
Images: (8, 3, 256, 256) torch.float32 | Masks: (8, 256, 256) torch.int64
Unique mask values in batch: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


Loss, optimization and metrics

In [8]:
# 1. Loss function: CrossEntropy + Dice Loss, ignoring the void class
ce_loss = nn.CrossEntropyLoss(ignore_index=VOID_CLASS)
dice_loss = smp.losses.DiceLoss(mode='multiclass', from_logits=True, ignore_index=VOID_CLASS)

def criterion(outputs, masks):
    return ce_loss(outputs, masks) + dice_loss(outputs, masks)

# 2. Optimizer: Adam with adaptive learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# 3. Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# 4. Metric: mean Intersection over Union computed from a confusion matrix
def update_confusion(confusion, preds, masks):
    valid = masks != VOID_CLASS
    idx = masks[valid] * NUM_CLASSES + preds[valid]
    confusion += torch.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
    return confusion

def miou_from_confusion(confusion):
    confusion = confusion.float()
    intersection = torch.diag(confusion)
    union = confusion.sum(0) + confusion.sum(1) - intersection
    iou = intersection / union.clamp(min=1)
    present = union[:VOID_CLASS] > 0   # only classes that appear (void excluded)
    return iou[:VOID_CLASS][present].mean().item(), iou

print("Loss function, optimizer, scheduler and metrics initialized.")

Loss function, optimizer, scheduler and metrics initialized.


Training and Validation

In [9]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, masks in tqdm(dataloader, leave=False):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(dataloader)


def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long, device=device)

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()

            confusion = update_confusion(confusion, outputs.argmax(dim=1), masks)

    miou, _ = miou_from_confusion(confusion)
    return running_loss / len(dataloader), miou

Training execution

In [ ]:
EPOCHS = 10
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_miou': []}

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_miou = validate_one_epoch(model, val_loader, criterion, DEVICE)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_miou'].append(val_miou)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val mIoU: {val_miou:.4f}")

    # Save best model parameters
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_segmentation_model.pth')
        print(" -> Model saved (best validation loss).")

  0%|          | 0/45 [00:00<?, ?it/s]

Prediction visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model.load_state_dict(torch.load('best_segmentation_model.pth', map_location=DEVICE))
model.eval()

# Quantitative evaluation on the full test set
confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long, device=DEVICE)
with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        confusion = update_confusion(confusion, model(images).argmax(dim=1), masks)
test_miou, per_class_iou = miou_from_confusion(confusion)
print(f"Test mIoU (void class excluded): {test_miou:.4f}")

# Qualitative evaluation: one batch from the test loader
images, masks = next(iter(test_loader))
images, masks = images.to(DEVICE), masks.to(DEVICE)

with torch.no_grad():
    preds = torch.argmax(model(images), dim=1)

# Plot comparison: Original Image vs Ground Truth vs Predicted Mask
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for i in range(3):
    # Undo the ImageNet normalization (same mean/std as the data pipeline)
    img = (images[i].cpu().permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title("Input Image")
    axes[i, 0].axis("off")

    # Same color scale for ground truth and prediction so colors are comparable
    axes[i, 1].imshow(masks[i].cpu().numpy(), cmap='tab20', vmin=0, vmax=NUM_CLASSES - 1)
    axes[i, 1].set_title("Ground Truth Mask")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(preds[i].cpu().numpy(), cmap='tab20', vmin=0, vmax=NUM_CLASSES - 1)
    axes[i, 2].set_title("Predicted Mask")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()